# Evaluación Formativa 2  
## Modelamiento Predictivo — Weather Dataset Australia

**Curso:** MCDI501: Estadística Computacional para la Toma de Decisiones  
**Programa:** Magíster en Ciencia de Datos e Inteligencia Artificial, UNAB  
**Grupo:** 7  
**Integrantes:** Alana Bermúdez, Gonzalo Mansilla y Eduardo Villalón  
**Dataset:** Weather Dataset Australia (`weatherAUS`)  
**Repositorio:** https://github.com/alanabermudezaballay/mcdi500_s1_grupo7  

## Objetivo del notebook

Este notebook desarrolla la preparación de datos para el modelamiento predictivo de `RainTomorrow`, integrando explícitamente resultados obtenidos en la Sumativa 1 y Sumativa 2. El objetivo es dejar una base reproducible para ajustar posteriormente un modelo de regresión logística, evaluar su desempeño predictivo y preparar la Sumativa 3.

# 1. Preparación de datos para modelamiento

Esta etapa utiliza como base los resultados de las evaluaciones anteriores:

## Resultados relevantes de Sumativa 1

- `Humidity3pm` mostró una diferencia relevante entre los grupos de `RainTomorrow`.
- La media de `Humidity3pm` fue mayor cuando `RainTomorrow = Yes` que cuando `RainTomorrow = No`.
- `RainToday` presentó asociación con `RainTomorrow`.
- `Pressure3pm` y `WindGustSpeed` fueron consideradas variables meteorológicas relevantes.
- La base presentaba valores faltantes, por lo que se trabajó con una estrategia de imputación.

## Resultados relevantes de Sumativa 2

- `Humidity3pm` fue robusta bajo bootstrap, prueba de permutación y análisis sin outliers.
- `Pressure3pm` mostró estabilidad bajo bootstrap y bajo exclusión de outliers.
- `WindGustSpeed` presentó sensibilidad a valores extremos, por lo que debe usarse con cautela.
- La asociación entre `RainToday` y `RainTomorrow` se mantuvo estable en el análisis de robustez.
- Se mantiene la advertencia metodológica de que la imputación por mediana puede reducir artificialmente la variabilidad y estrechar intervalos de confianza.

Con base en estos resultados, se seleccionan inicialmente las siguientes variables predictoras:

- `RainToday`
- `Humidity3pm`
- `Pressure3pm`
- `WindGustSpeed`

La variable objetivo será:

- `RainTomorrow`

In [23]:
# Librerías base
import pandas as pd
import numpy as np
from pathlib import Path

# Configuración general
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

RANDOM_STATE = 42

In [24]:
from pathlib import Path
import pandas as pd

# Rutas candidatas para encontrar el dataset limpio usado en S1/S2
rutas_candidatas = [
    Path("weatherAUS_limpio.csv"),
    Path("S2/weatherAUS_limpio.csv"),
    Path("EstadisticaComputacional/S2/weatherAUS_limpio.csv"),
    Path("../S2/weatherAUS_limpio.csv"),
    Path("../EstadisticaComputacional/S2/weatherAUS_limpio.csv"),
    Path("../../EstadisticaComputacional/S2/weatherAUS_limpio.csv"),
]

ruta_dataset = None

print("Buscando dataset en rutas candidatas:\n")

for ruta in rutas_candidatas:
    print(f"{ruta} -> existe: {ruta.exists()}")
    if ruta.exists():
        ruta_dataset = ruta
        break

if ruta_dataset is None:
    raise FileNotFoundError(
        "No se encontró weatherAUS_limpio.csv. "
        "Revisa si el archivo está en EstadisticaComputacional/S2 o copia el archivo a la carpeta F2."
    )

df = pd.read_csv(ruta_dataset)

print("\nDataset cargado correctamente")
print(f"Ruta usada: {ruta_dataset}")
print(f"Filas: {df.shape[0]:,}")
print(f"Columnas: {df.shape[1]:,}")

df.head()

Buscando dataset en rutas candidatas:

weatherAUS_limpio.csv -> existe: False
S2\weatherAUS_limpio.csv -> existe: False
EstadisticaComputacional\S2\weatherAUS_limpio.csv -> existe: False
..\S2\weatherAUS_limpio.csv -> existe: False
..\EstadisticaComputacional\S2\weatherAUS_limpio.csv -> existe: False
..\..\EstadisticaComputacional\S2\weatherAUS_limpio.csv -> existe: True

Dataset cargado correctamente
Ruta usada: ..\..\EstadisticaComputacional\S2\weatherAUS_limpio.csv
Filas: 145,460
Columnas: 23


,Date,Location,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustDir,WindGustSpeed,WindDir9am,WindDir3pm,WindSpeed9am,WindSpeed3pm,Humidity9am,Humidity3pm,Pressure9am,Pressure3pm,Cloud9am,Cloud3pm,Temp9am,Temp3pm,RainToday,RainTomorrow
0,2008-12-01,Albury,13.4000,22.9000,0.6000,4.8000,8.4000,W,44.0000,W,WNW,20.0000,24.0000,71.0000,22.0000,"1,007.7000","1,007.1000",8.0000,5.0000,16.9000,21.8000,No,No
1,2008-12-02,Albury,7.4000,25.1000,0.0000,4.8000,8.4000,WNW,44.0000,NNW,WSW,4.0000,22.0000,44.0000,25.0000,"1,010.6000","1,007.8000",5.0000,5.0000,17.2000,24.3000,No,No
2,2008-12-03,Albury,12.9000,25.7000,0.0000,4.8000,8.4000,WSW,46.0000,W,WSW,19.0000,26.0000,38.0000,30.0000,"1,007.6000","1,008.7000",5.0000,2.0000,21.0000,23.2000,No,No
3,2008-12-04,Albury,9.2000,28.0000,0.0000,4.8000,8.4000,NE,24.0000,SE,E,11.0000,9.0000,45.0000,16.0000,"1,017.6000","1,012.8000",5.0000,5.0000,18.1000,26.5000,No,No
4,2008-12-05,Albury,17.5000,32.3000,1.0000,4.8000,8.4000,W,41.0000,ENE,NW,7.0000,20.0000,82.0000,33.0000,"1,010.8000","1,006.0000",7.0000,8.0000,17.8000,29.7000,No,No


In [25]:
from pathlib import Path
import os

print("Carpeta actual de ejecución:")
print(Path.cwd())

print("\nArchivos/carpetas visibles desde esta ubicación:")
for item in Path.cwd().iterdir():
    print("-", item.name)

Carpeta actual de ejecución:
c:\MCDI\mcdi500_s1_grupo7\ESTADÍSTICA COMPUTACIONAL PARA LA TOMA DE DECISIONES\F2

Archivos/carpetas visibles desde esta ubicación:
- datos_preparados


In [26]:
# Revisión de columnas disponibles
df.columns.tolist()

['Date',
 'Location',
 'MinTemp',
 'MaxTemp',
 'Rainfall',
 'Evaporation',
 'Sunshine',
 'WindGustDir',
 'WindGustSpeed',
 'WindDir9am',
 'WindDir3pm',
 'WindSpeed9am',
 'WindSpeed3pm',
 'Humidity9am',
 'Humidity3pm',
 'Pressure9am',
 'Pressure3pm',
 'Cloud9am',
 'Cloud3pm',
 'Temp9am',
 'Temp3pm',
 'RainToday',
 'RainTomorrow']

In [27]:
# Variables requeridas para esta etapa
predictoras = ["RainToday", "Humidity3pm", "Pressure3pm", "WindGustSpeed"]
objetivo = "RainTomorrow"

columnas_requeridas = predictoras + [objetivo]

faltantes_columnas = [col for col in columnas_requeridas if col not in df.columns]

if faltantes_columnas:
    raise ValueError(f"Faltan columnas requeridas en el dataset: {faltantes_columnas}")

print("Todas las columnas requeridas están disponibles:")
print(columnas_requeridas)

Todas las columnas requeridas están disponibles:
['RainToday', 'Humidity3pm', 'Pressure3pm', 'WindGustSpeed', 'RainTomorrow']


In [28]:
# Base reducida para modelamiento
df_modelo = df[columnas_requeridas].copy()

print("Dimensión base de modelamiento:")
print(df_modelo.shape)

df_modelo.head()

Dimensión base de modelamiento:
(145460, 5)


,RainToday,Humidity3pm,Pressure3pm,WindGustSpeed,RainTomorrow
0,No,22.0000,"1,007.1000",44.0000,No
1,No,25.0000,"1,007.8000",44.0000,No
2,No,30.0000,"1,008.7000",46.0000,No
3,No,16.0000,"1,012.8000",24.0000,No
4,No,33.0000,"1,006.0000",41.0000,No


In [29]:
# Revisión de valores faltantes antes del tratamiento
faltantes = df_modelo.isna().sum().to_frame("faltantes")
faltantes["porcentaje"] = (faltantes["faltantes"] / len(df_modelo) * 100).round(2)

faltantes

,faltantes,porcentaje
RainToday,0,0.0000
Humidity3pm,0,0.0000
Pressure3pm,0,0.0000
WindGustSpeed,0,0.0000
RainTomorrow,0,0.0000


In [30]:
# Conversión robusta de variables binarias Yes/No a 1/0

def convertir_binaria_robusta(serie):
    """
    Convierte variables binarias tipo Yes/No, yes/no, 1/0 a valores 1/0.
    Mantiene NaN cuando no existe valor.
    """
    s = serie.astype("string").str.strip().str.lower()
    
    return s.map({
        "yes": 1,
        "no": 0,
        "1": 1,
        "0": 0
    })

df_modelo["RainToday"] = convertir_binaria_robusta(df_modelo["RainToday"])
df_modelo["RainTomorrow"] = convertir_binaria_robusta(df_modelo["RainTomorrow"])

print("Valores únicos RainToday después de conversión:")
print(df_modelo["RainToday"].value_counts(dropna=False))

print("\nValores únicos RainTomorrow después de conversión:")
print(df_modelo["RainTomorrow"].value_counts(dropna=False))

Valores únicos RainToday después de conversión:
RainToday
0    113580
1     31880
Name: count, dtype: int64

Valores únicos RainTomorrow después de conversión:
RainTomorrow
0    113583
1     31877
Name: count, dtype: int64


## Tratamiento de valores faltantes

Se aplica imputación simple para mantener la mayor cantidad posible de observaciones. Las variables numéricas se imputan con la mediana, criterio consistente con las etapas previas y más robusto frente a valores extremos. La variable binaria `RainToday` se imputa con la moda si existieran valores faltantes.

La variable objetivo `RainTomorrow` no se imputa; si existieran registros sin objetivo, se eliminan para evitar introducir etiquetas artificiales.

In [31]:
# Imputación simple
variables_numericas = ["Humidity3pm", "Pressure3pm", "WindGustSpeed"]
variables_binarias = ["RainToday"]

# Imputación de variables numéricas con mediana
for col in variables_numericas:
    mediana = df_modelo[col].median()
    df_modelo[col] = df_modelo[col].fillna(mediana)
    print(f"{col}: imputado con mediana = {mediana:.4f}")

# Imputación de variable binaria con moda
for col in variables_binarias:
    moda = df_modelo[col].mode(dropna=True)[0]
    df_modelo[col] = df_modelo[col].fillna(moda)
    print(f"{col}: imputado con moda = {moda}")

# Eliminar registros sin variable objetivo
filas_antes = len(df_modelo)
df_modelo = df_modelo.dropna(subset=["RainTomorrow"])
filas_despues = len(df_modelo)

print(f"\nFilas eliminadas por objetivo faltante: {filas_antes - filas_despues}")
print(f"Dimensión final base modelamiento: {df_modelo.shape}")

Humidity3pm: imputado con mediana = 52.0000
Pressure3pm: imputado con mediana = 1015.2000
WindGustSpeed: imputado con mediana = 39.0000
RainToday: imputado con moda = 0

Filas eliminadas por objetivo faltante: 0
Dimensión final base modelamiento: (145460, 5)


In [32]:
# Confirmación de faltantes después del tratamiento
df_modelo.isna().sum()

RainToday        0
Humidity3pm      0
Pressure3pm      0
WindGustSpeed    0
RainTomorrow     0
dtype: int64

In [33]:
# Separación de variables predictoras y objetivo
X = df_modelo[predictoras].copy()
y = df_modelo[objetivo].copy().astype("int64")

print("Dimensión X:", X.shape)
print("Dimensión y:", y.shape)

print("\nDistribución de la variable objetivo:")
print(y.value_counts(normalize=True).rename("proporción"))

print("\nConteo absoluto de la variable objetivo:")
print(y.value_counts())

Dimensión X: (145460, 4)
Dimensión y: (145460,)

Distribución de la variable objetivo:
RainTomorrow
0   0.7809
1   0.2191
Name: proporción, dtype: float64

Conteo absoluto de la variable objetivo:
RainTomorrow
0    113583
1     31877
Name: count, dtype: int64


## División train/test

Se divide la base en entrenamiento y prueba usando una proporción 70/30. Para conservar la proporción de la variable objetivo `RainTomorrow`, la partición se realiza de forma estratificada manualmente, separando los índices de cada clase antes de seleccionar entrenamiento y prueba.

In [34]:
# División train/test 70/30 estratificada sin scikit-learn

rng = np.random.default_rng(RANDOM_STATE)

train_indices = []
test_indices = []

for clase in sorted(y.unique()):
    # Se usa .copy() para evitar error de arreglo read-only
    indices_clase = y[y == clase].index.to_numpy().copy()
    rng.shuffle(indices_clase)
    
    n_test = int(round(len(indices_clase) * 0.30))
    
    test_indices.extend(indices_clase[:n_test])
    train_indices.extend(indices_clase[n_test:])

train_indices = np.array(train_indices)
test_indices = np.array(test_indices)

X_train = X.loc[train_indices].copy()
X_test = X.loc[test_indices].copy()
y_train = y.loc[train_indices].copy()
y_test = y.loc[test_indices].copy()

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

print("\nDistribución objetivo en entrenamiento:")
print(y_train.value_counts(normalize=True).rename("proporción"))

print("\nDistribución objetivo en prueba:")
print(y_test.value_counts(normalize=True).rename("proporción"))

X_train: (101822, 4)
X_test : (43638, 4)
y_train: (101822,)
y_test : (43638,)

Distribución objetivo en entrenamiento:
RainTomorrow
0   0.7809
1   0.2191
Name: proporción, dtype: float64

Distribución objetivo en prueba:
RainTomorrow
0   0.7809
1   0.2191
Name: proporción, dtype: float64


## Estandarización de variables numéricas

Las variables continuas `Humidity3pm`, `Pressure3pm` y `WindGustSpeed` se estandarizan usando la media y desviación estándar calculadas solo sobre el conjunto de entrenamiento. Luego, esos mismos parámetros se aplican al conjunto de prueba.

Este procedimiento evita fuga de información desde el conjunto de prueba hacia el entrenamiento. La variable `RainToday` no se estandariza porque ya está codificada como binaria 0/1.

In [35]:
# Estandarización manual de variables numéricas

variables_numericas = ["Humidity3pm", "Pressure3pm", "WindGustSpeed"]

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

# Parámetros calculados solo con entrenamiento
medias_train = X_train[variables_numericas].mean()
desvios_train = X_train[variables_numericas].std(ddof=0)

# Evitar división por cero
desvios_train = desvios_train.replace(0, 1)

# Aplicar estandarización
X_train_scaled[variables_numericas] = (
    X_train[variables_numericas] - medias_train
) / desvios_train

X_test_scaled[variables_numericas] = (
    X_test[variables_numericas] - medias_train
) / desvios_train

print("Medias usadas para estandarizar:")
print(medias_train)

print("\nDesviaciones estándar usadas para estandarizar:")
print(desvios_train)

print("\nResumen de X_train_scaled:")
X_train_scaled.describe().T

Medias usadas para estandarizar:
Humidity3pm        51.5715
Pressure3pm     1,015.2472
WindGustSpeed      39.9697
dtype: float64

Desviaciones estándar usadas para estandarizar:
Humidity3pm     20.4708
Pressure3pm      6.6674
WindGustSpeed   13.1611
dtype: float64

Resumen de X_train_scaled:


,count,mean,std,min,25%,50%,75%,max
RainToday,"101,822.0000",0.2193,0.4138,0.0000,0.0000,0.0000,0.0000,1.0000
Humidity3pm,"101,822.0000",0.0000,1.0000,-2.5193,-0.7118,0.0209,0.6560,2.3657
Pressure3pm,"101,822.0000",0.0000,1.0000,-5.7214,-0.6370,-0.0071,0.6228,3.6525
WindGustSpeed,"101,822.0000",-0.0000,1.0000,-2.5811,-0.6815,-0.0737,0.4582,7.2205


In [36]:
# Guardar archivos preparados para trazabilidad y uso posterior

salida_dir = Path("datos_preparados")
salida_dir.mkdir(exist_ok=True)

X_train_scaled.to_csv(salida_dir / "X_train_scaled.csv", index=False)
X_test_scaled.to_csv(salida_dir / "X_test_scaled.csv", index=False)
y_train.to_csv(salida_dir / "y_train.csv", index=False)
y_test.to_csv(salida_dir / "y_test.csv", index=False)

# Guardar parámetros de estandarización usados sobre el conjunto de entrenamiento
parametros_estandarizacion = pd.DataFrame({
    "media_train": medias_train,
    "desvio_train": desvios_train
})

parametros_estandarizacion.to_csv(
    salida_dir / "parametros_estandarizacion.csv",
    index=True
)

print("Archivos guardados en carpeta:", salida_dir)
print("Archivos generados:")
for archivo in salida_dir.iterdir():
    print("-", archivo.name)

Archivos guardados en carpeta: datos_preparados
Archivos generados:
- parametros_estandarizacion.csv
- X_test_scaled.csv
- X_train_scaled.csv
- y_test.csv
- y_train.csv


## Cierre de preparación de datos

La base de modelamiento quedó preparada para ajustar un modelo de regresión logística orientado a predecir `RainTomorrow`.

Se utilizaron cuatro variables predictoras seleccionadas desde los resultados previos de S1 y S2:

- `RainToday`, por su asociación robusta con `RainTomorrow`.
- `Humidity3pm`, por su diferencia significativa y robusta entre días con y sin lluvia futura.
- `Pressure3pm`, por su estabilidad bajo bootstrap y análisis de robustez.
- `WindGustSpeed`, por su relevancia meteorológica, aunque documentando su sensibilidad a outliers.

La base fue dividida en entrenamiento y prueba en proporción 70/30, con estratificación por la variable objetivo. Las variables numéricas fueron estandarizadas usando únicamente los parámetros del conjunto de entrenamiento, evitando fuga de información hacia el conjunto de prueba.

Los objetos `X_train_scaled`, `X_test_scaled`, `y_train` y `y_test` quedan disponibles para la etapa siguiente de ajuste del modelo de regresión logística. Además, se guardaron archivos `.csv` en la carpeta `datos_preparados` para asegurar trazabilidad y facilitar el trabajo colaborativo.